In [1]:
import pandas as pd
import time
import requests
import os 
from dotenv import load_dotenv
load_dotenv()
import ollama
from tqdm import tqdm

# --- CONFIGURACIÓN ---
MODELO = "gemma3:4b"
ARCHIVO_ENTRADA = "./datasets/dataset_steam.csv"
ARCHIVO_MAESTRO = "./datasets/juegos_maestro2.csv" # Usaremos un solo archivo persistente
PAUSA_STEAM = 1.5 

if not os.path.exists('./datasets'): os.makedirs('./datasets')

# --- FUNCIONES DE APOYO (Mantenemos las anteriores) ---

def obtener_info_juego(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 429:
            time.sleep(30)
            return None
        data = response.json()
        if data and data.get(str(appid), {}).get('success'):
            details = data[str(appid)]['data']
            genres_list = details.get('genres', [])
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': details.get('developers', ["Desconocido"])[0],
                'genres_steam': ", ".join([g['description'] for g in genres_list]) if genres_list else "Sin Género",
                'price': 0 if details.get('is_free') else (details.get('price_overview', {}).get('final', 0) / 100),
                'country': None, # Inicializamos vacío
                'genre': None    # Inicializamos vacío
            }
    except: return None

def obtener_pais_empresa(empresa):
    prompt = f"Indica el país de la sede de '{empresa}'. Responde SOLO el nombre del país en español."
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip().replace(".", "")
    except: return "Error"

def obtener_genero_limpio(combo):
    prompt = f"De esta lista: '{combo}', dime el género principal (una palabra). Ej: Acción, RPG, Aventura."
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip().split('\n')[0].replace(".", "")
    except: return "Otros"

# --- PROCESO CON REANUDACIÓN TOTAL ---

# 1. Cargar o Crear el Dataset Maestro
if os.path.exists(ARCHIVO_MAESTRO):
    df_maestro = pd.read_csv(ARCHIVO_MAESTRO)
    print(f"🔄 Cargando dataset maestro con {len(df_maestro)} registros.")
else:
    df_maestro = pd.DataFrame(columns=['appid', 'name', 'developer', 'genres_steam', 'price', 'country', 'genre'])
    print("🆕 Creado nuevo dataset maestro.")

# 2. PASO 1: Steam (Solo los que faltan en el maestro)
df_inicial = pd.read_csv(ARCHIVO_ENTRADA)
ids_totales = df_inicial["appid"].unique()
ids_faltantes_steam = [aid for aid in ids_totales if aid not in df_maestro['appid'].values]

if ids_faltantes_steam:
    print(f"--- PASO 1: Steam ({len(ids_faltantes_steam)} pendientes) ---")
    nuevos_datos = []
    for i, appid in enumerate(tqdm(ids_faltantes_steam, desc="Steam")):
        info = obtener_info_juego(appid)
        if info: nuevos_datos.append(info)
        if (i + 1) % 50 == 0: # Guardado frecuente
            df_maestro = pd.concat([df_maestro, pd.DataFrame(nuevos_datos)], ignore_index=True)
            df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
            nuevos_datos = []
        time.sleep(PAUSA_STEAM)

    df_maestro = pd.concat([df_maestro, pd.DataFrame(nuevos_datos)], ignore_index=True)
    df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# 3. PASO 2: Países (Solo si 'country' es nulo)
# Buscamos desarrolladores cuyos juegos no tengan país asignado aún
mask_no_pais = df_maestro['country'].isna() | (df_maestro['country'] == "Error")
if mask_no_pais.any():
    print(f"\n--- PASO 2: Países pendientes ---")
    devs_pendientes = df_maestro[mask_no_pais]['developer'].unique()
    mapa_paises = {}
    for dev in tqdm(devs_pendientes, desc="Ollama Países"):
        pais = obtener_pais_empresa(dev)
        mapa_paises[dev] = pais

        # Actualizamos el DF y guardamos cada vez para no perder progreso
        df_maestro.loc[df_maestro['developer'] == dev, 'country'] = pais
        df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# 4. PASO 3: Géneros (Solo si 'genre' es nulo)
mask_no_genero = df_maestro['genre'].isna() | (df_maestro['genre'] == "Otros")
if mask_no_genero.any():
    print(f"\n--- PASO 3: Géneros pendientes ---")
    combos_pendientes = df_maestro[mask_no_genero]['genres_steam'].unique()
    for combo in tqdm(combos_pendientes, desc="Ollama Géneros"):
        gen_limpio = obtener_genero_limpio(combo)

        # Actualizamos el DF y guardamos cada vez para no perder progreso
        df_maestro.loc[df_maestro['genres_steam'] == combo, 'genre'] = gen_limpio
        df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

print(f"\n✅ Todo procesado. Archivo final: {ARCHIVO_MAESTRO}")


🆕 Creado nuevo dataset maestro.
--- PASO 1: Steam (12408 pendientes) ---


Steam:   0%|          | 33/12408 [01:00<6:16:54,  1.83s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
df_maestro2 = pd.read_csv("./datasets/juegos_maestro2.csv")


In [ ]:
# --- PROCESO CON REANUDACIÓN TOTAL Y PARALELISMO ---

import pandas as pd
import requests
import time
import ollama
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- CONFIGURACIÓN ---
MODELO = "gemma3:4b"  # Recomendado por velocidad
ARCHIVO_ENTRADA = "./datasets/dataset_steam.csv"
ARCHIVO_MAESTRO = "./datasets/juegos_maestro2.csv"

MAX_THREADS_STEAM = 3   # Máximo 2-3 para evitar baneos de Steam
MAX_THREADS_OLLAMA = 4  # Ajusta según tu GPU/RAM (4-8 suele ser ideal)
PAUSA_STEAM = 1.0       # Respiro extra entre peticiones concurrentes

if not os.path.exists('./datasets'): os.makedirs('./datasets')

# --- FUNCIONES DE APOYO ---

def obtener_info_juego(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 429:
            time.sleep(20)
            return None
        data = response.json()
        if data and data.get(str(appid), {}).get('success'):
            details = data[str(appid)]['data']
            genres_list = details.get('genres', [])
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': details.get('developers', ["Desconocido"])[0],
                'genres_steam': ", ".join([g['description'] for g in genres_list]) if genres_list else "Sin Género",
                'price': 0 if details.get('is_free') else (details.get('price_overview', {}).get('final', 0) / 100),
                'country': None,
                'genre': None
            }
    except: return None

def obtener_pais_worker(dev):
    """Worker para procesar países en paralelo."""
    prompt = f"""Empresa: {dev}. Eres un buscador de sedes corporativas.
    Tarea: Indica el país donde se encuentra la sede principal de la empresa: '{dev}'.
    
    Reglas críticas de respuesta:
    1. Responde ÚNICAMENTE con el nombre del país en español.
    2. NO escribas frases como "La sede está en..." o "El país es...".
    3. NO uses puntos finales ni signos de puntuación.
    4. NO des ninguna información adicional, ni ciudades, ni fechas. Lo que nos interesa es dónde están situados los HQ de la empresa.
    5. Si no conoces el país, responde exclusivamente con la palabra: Desconocido.
    6. Responde con exactamente UNA palabra.
    """
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        pais = response['message']['content'].strip().split('\n')[0].replace(".", "")
        return dev, pais
    except: return dev, "Error"

def obtener_genero_worker(nombre_juego):
    """Función que será ejecutada por cada hilo."""
    # Lista base para consistencia (puedes ampliarla)
    generos_referencia = "Acción, RPG, Estrategia, Aventura, Simulación, Deportes, Puzzle, Shooter"
    
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: Define el género principal del juego '{nombre_juego}'.
    Reglas:
    1. Responde con exactamente UNA palabra.
    2. Género genérico, NO específico.
    3. Usa esta lista como preferencia: [{generos_referencia}].
    4. NO uses puntos ni explicaciones.
    """

    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        genero = response['message']['content'].strip().split('\n')[0].replace(".", "").capitalize()
        return nombre_juego, genero
    except Exception as e:
        return nombre_juego, "Error"

# --- PROCESO PRINCIPAL ---

# 1. Cargar o Crear Dataset Maestro
if os.path.exists(ARCHIVO_MAESTRO):
    df_maestro = pd.read_csv(ARCHIVO_MAESTRO)
else:
    df_maestro = pd.DataFrame(columns=['appid', 'name', 'developer', 'genres_steam', 'price', 'country', 'genre'])



# 2. PARALELISMO PASO 1: Steam
df_inicial = pd.read_csv(ARCHIVO_ENTRADA)
ids_totales = df_inicial["appid"].unique()
ids_faltantes = [aid for aid in ids_totales if aid not in df_maestro['appid'].values]



if ids_faltantes:
    print(f"--- PASO 1: Steam ({len(ids_faltantes)} hilos: {MAX_THREADS_STEAM}) ---")
    with ThreadPoolExecutor(max_workers=MAX_THREADS_STEAM) as executor:
        futures = {executor.submit(obtener_info_juego, aid): aid for aid in ids_faltantes}
        nuevos_registros = []
        for f in tqdm(as_completed(futures), total=len(futures), desc="Steam"):
            res = f.result()
            if res:
                nuevos_registros.append(res)
            
            # Guardado incremental cada 20 resultados para no perder nada
            if len(nuevos_registros) >= 20:
                df_maestro = pd.concat([df_maestro, pd.DataFrame(nuevos_registros)], ignore_index=True)
                df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
                nuevos_registros = []
            time.sleep(PAUSA_STEAM / MAX_THREADS_STEAM) # Distribución de carga
        
        if nuevos_registros:
            df_maestro = pd.concat([df_maestro, pd.DataFrame(nuevos_registros)], ignore_index=True)
            df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

df_maestro.drop(columns=['country', 'genre'], inplace=True, errors='ignore')

# 3. PARALELISMO PASO 2: Países
print(f"\n--- REPROCESANDO PASO 2: Ollama Países (Todos los registros) ---")

# Obtenemos todos los desarrolladores únicos (sin filtrar)
devs_totales = df_maestro['developer'].unique()
print(f"Total de desarrolladores a procesar: {len(devs_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = [executor.submit(obtener_pais_worker, d) for d in devs_totales]
    
    for f in tqdm(as_completed(futures), total=len(futures), desc="Países (Total)"):
        dev, pais = f.result()
        # Sobreescribimos el valor anterior para ese desarrollador
        df_maestro.loc[df_maestro['developer'] == dev, 'country'] = pais
        
        # Guardado incremental de seguridad cada 20 devs
        if len(futures) % 20 == 0:
            df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# Guardado final de este paso
df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# 4. PARALELISMO PASO 3: Géneros (Optimizado)
print(f"\n--- REPROCESANDO PASO 3: Ollama Géneros (Todos los registros) ---")

# Obtenemos todos los combos únicos de etiquetas de Steam presentes en el dataset
combos_totales = df_maestro['genres_steam'].unique()
print(f"Total de combos únicos a reclasificar: {len(combos_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    # Usamos un diccionario para saber qué combo corresponde a qué hilo
    futures = {executor.submit(obtener_genero_worker, c): c for c in combos_totales}
    
    contador = 0
    for f in tqdm(as_completed(futures), total=len(futures), desc="Géneros (Total)"):
        combo_original = futures[f]
        try:
            _, genero_limpio = f.result()
            
            # Aplicamos el nuevo género a todas las filas que tengan ese combo
            df_maestro.loc[df_maestro['genres_steam'] == combo_original, 'genre'] = genero_limpio
            
            # Guardado incremental cada 15 combos
            contador += 1
            if contador >= 15:
                df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
                contador = 0
                
        except Exception as e:
            print(f"Error en combo {combo_original}: {e}")

# Guardado final absoluto
df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
print(f"\n✅ Reprocesamiento completo. Datos actualizados en {ARCHIVO_MAESTRO}")

--- PASO 1: Steam (262 hilos: 3) ---


Steam: 100%|██████████| 262/262 [05:25<00:00,  1.24s/it]



--- REPROCESANDO PASO 2: Ollama Países (Todos los registros) ---
Total de desarrolladores a procesar: 7371


Países (Total):  86%|████████▋ | 6363/7371 [2:07:22<21:16,  1.27s/it]  